# 11 - Independent paired baseline and pruning seeds

Trains five independent M0 baselines, induces a distinct magnitude mask from each baseline, fine-tunes prune50/prune80, and reports paired per-class effects and mask overlap.

**Safety:** this notebook writes only new files under `results/tables/comnet/` and does not overwrite archived manuscript result tables.

In [ ]:
# Colab/bootstrap cell: no tokens or credentials are required.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import os, sys, json, time
from pathlib import Path

REPO = Path('/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression')
if not REPO.exists():
    # Local/Jupyter fallback: run the notebook from the repository root.
    REPO = Path.cwd()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.config import CFG, PATHS, set_all_seeds
set_all_seeds(CFG['anchor_seed'])

OUT_TABLE = PATHS.tables('comnet')
OUT_TABLE.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Outputs:', OUT_TABLE)


## Configuration
This is the principal heavy rerun. It trains independent baselines and induces a different deterministic magnitude mask from each baseline.

In [ ]:
DATASET = 'ciciot2023'
ARCH = 'cnn1d'
ARCH_KW = {'channels': (64, 128)}
SEEDS = list(CFG['seeds'])
RUN_PRUNE50 = True
RUN_PRUNE80 = True
SAVE_CHECKPOINTS = True
PRACTICAL_LOSS = 0.10

RUN_LABEL = 'independent_paired_v1'


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score

from src.data import load_raw, clean, temporal_within_capture_split
from src.train import train_model, predict, per_class_recall_table
from src.compression import prune_and_finetune
from src.comnet_audit import assign_validation_tiers, mask_jaccard, environment_record, write_json

df = clean(load_raw(DATASET, subsample=True, seed=CFG['anchor_seed']), DATASET)
splits = temporal_within_capture_split(df, CFG['anchor_seed'])


## Independent paired baseline -> pruning runs

In [ ]:
baseline_val, baseline_test, compressed_test = {}, {}, {}
run_summary, pruned_models = [], {}

for seed in SEEDS:
    print(f'\n===== seed {seed} =====')
    m0, info = train_model(
        ARCH, df, DATASET, splits, seed,
        epochs=40, patience=6, batch_size=4096, lr=1e-3,
        compression='M0_paired', arch_kwargs=ARCH_KW,
        save=SAVE_CHECKPOINTS, verbose=True,
    )
    le, scaler, feat_cols = info['label_encoder'], info['scaler'], info['feat_cols']
    yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val')
    yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
    baseline_val[seed] = per_class_recall_table(yv, pv, le).set_index('label')['recall']
    baseline_test[seed] = per_class_recall_table(yt, pt, le).set_index('label')['recall']
    run_summary.append({'seed': seed, 'cell': 'M0',
                        'val_macro_f1': f1_score(yv,pv,average='macro'),
                        'test_macro_f1': f1_score(yt,pt,average='macro')})

    for amount, cell in [(0.50,'prune50'), (0.80,'prune80')]:
        if (cell == 'prune50' and not RUN_PRUNE50) or (cell == 'prune80' and not RUN_PRUNE80):
            continue
        mp, lep, scp = prune_and_finetune(
            m0, df, DATASET, splits, seed, amount,
            ft_epochs=8, batch_size=4096, lr=5e-4, arch=ARCH, verbose=True,
        )
        ytc, ptc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
        compressed_test[(cell,seed)] = per_class_recall_table(ytc, ptc, le).set_index('label')['recall']
        run_summary.append({'seed': seed, 'cell': cell,
                            'val_macro_f1': np.nan,
                            'test_macro_f1': f1_score(ytc,ptc,average='macro')})
        if cell == 'prune80':
            pruned_models[seed] = mp.cpu()
        if SAVE_CHECKPOINTS:
            path = PATHS.model(DATASET, ARCH, f'{cell}_paired', seed)
            torch.save({'state_dict': mp.state_dict(), 'classes': list(le.classes_),
                        'feat_cols': feat_cols, 'scaler_mean': scaler.mean_,
                        'scaler_scale': scaler.scale_}, path)
            print('saved', path)

pd.DataFrame(run_summary).to_csv(OUT_TABLE / 'paired_seed_run_summary.csv', index=False)


## Validation-frozen tiers and paired per-class effects

In [ ]:
val_R = pd.DataFrame(baseline_val)
tiers = assign_validation_tiers(val_R)
tiers.to_csv(OUT_TABLE / 'paired_validation_defined_tiers.csv')

test_R0 = pd.DataFrame(baseline_test)
rows = []
for (cell, seed), rc in compressed_test.items():
    r0 = test_R0[seed]
    common = r0.index.intersection(rc.index)
    for cls in common:
        loss = float(r0.loc[cls] - rc.loc[cls])
        band = float(tiers.loc[cls, 'validation_2sd_band']) if cls in tiers.index else np.nan
        rows.append({'seed': seed, 'cell': cell, 'class': cls,
                     'M0_test_recall': float(r0.loc[cls]),
                     'compressed_test_recall': float(rc.loc[cls]),
                     'recall_loss': loss,
                     'validation_tier': tiers.loc[cls, 'validation_tier'],
                     'validation_2sd_band': band,
                     'crosses_validation_band': bool(loss > band),
                     'practically_material': bool(loss >= PRACTICAL_LOSS),
                     'material_and_beyond_band': bool((loss > band) and (loss >= PRACTICAL_LOSS))})

paired = pd.DataFrame(rows)
paired.to_csv(OUT_TABLE / 'paired_seed_per_class_effects.csv', index=False)
summary = paired.groupby(['cell','class']).agg(
    mean_recall_loss=('recall_loss','mean'),
    sd_recall_loss=('recall_loss','std'),
    affected_frequency=('material_and_beyond_band','mean'),
    n=('seed','nunique'),
).reset_index()
summary.to_csv(OUT_TABLE / 'paired_seed_per_class_summary.csv', index=False)
display(summary.sort_values(['cell','mean_recall_loss'], ascending=[True,False]).head(40))


## Mask overlap across independently trained baselines

In [ ]:
jac = []
keys = sorted(pruned_models)
for i, a in enumerate(keys):
    for b in keys[i+1:]:
        tab = mask_jaccard(pruned_models[a], pruned_models[b])
        g = tab[tab['tensor']=='GLOBAL']
        jac.append({'seed_a':a, 'seed_b':b,
                    'global_nonzero_mask_jaccard': float(g.iloc[0]['jaccard_nonzero']) if len(g) else np.nan})
mask_overlap = pd.DataFrame(jac)
mask_overlap.to_csv(OUT_TABLE / 'paired_seed_mask_jaccard.csv', index=False)
display(mask_overlap)


## Submission-ready summary

In [ ]:
aggregate = pd.DataFrame(run_summary).pivot(index='seed', columns='cell', values='test_macro_f1')
aggregate.to_csv(OUT_TABLE / 'paired_seed_macro_f1_wide.csv')

def aggregate_stats(x):
    x = pd.Series(x).dropna().astype(float)
    return pd.Series({'n':len(x), 'mean':x.mean(), 'sd':x.std(ddof=1),
                      'min':x.min(), 'max':x.max()})
macro_summary = pd.DataFrame(run_summary).groupby('cell')['test_macro_f1'].apply(aggregate_stats).unstack()
macro_summary.to_csv(OUT_TABLE / 'paired_seed_macro_f1_summary.csv')
write_json(OUT_TABLE / 'paired_seed_run_environment.json', {
    'run_label': RUN_LABEL,
    'seeds': SEEDS,
    'practical_recall_loss_threshold': PRACTICAL_LOSS,
    'environment': environment_record(),
})
display(aggregate)
display(macro_summary.round(4))
print('Use these paired results to replace the fixed-mask sensitivity claim only after all seeds complete successfully.')
